<a href="https://colab.research.google.com/github/suhanisingh5617-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suhanisingh5617-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row in my slice = one content page, on one day. I'll use the daily fact table joined with the pages dimension table, filtered to a mid-panel month (month=2026-03) to develop my logic, keeping the _sample (June 2026) table sealed for later testing.

In [6]:
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset

login(token=userdata.get("HF_TOKEN"))

ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True)

# Just peek at one row to see real column names
first_row = next(iter(ds['train']))
print(first_row)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_users': 0, 'ga4_engaged_sessions': 0, 'ga4_total_engagement_sec': 0, 'sessions_organic': 0, 'sessions_direct': 0, 'sessions_referral': 0, 'sessions_social': 0, 'sessions_paid': 0, 'sessions_ai': 0, 'ai_chatgpt': 0, 'ai_perplexity': 0, 'ai_gemini': 0, 'ai_copilot': 0, 'ai_claude': 0, 'ai_meta': 0, 'ai_other': 0, 'scroll_events': 0}


In [7]:
import json
print(json.dumps(first_row, indent=2, default=str))

{
  "report_date": "2025-01-27",
  "client_hash_id": "client_9958f0a7ae1df715",
  "content_hash_id": "content_3b70a18ea133b2bb",
  "client_has_gsc": true,
  "client_has_ga4": true,
  "gsc_data_available": true,
  "ga4_data_available": false,
  "gsc_impressions": 30,
  "gsc_clicks": 0,
  "gsc_sum_position": 115,
  "gsc_avg_position": 3.8333333333333335,
  "ga4_pageviews": 0,
  "ga4_sessions": 0,
  "ga4_users": 0,
  "ga4_engaged_sessions": 0,
  "ga4_total_engagement_sec": 0,
  "sessions_organic": 0,
  "sessions_direct": 0,
  "sessions_referral": 0,
  "sessions_social": 0,
  "sessions_paid": 0,
  "sessions_ai": 0,
  "ai_chatgpt": 0,
  "ai_perplexity": 0,
  "ai_gemini": 0,
  "ai_copilot": 0,
  "ai_claude": 0,
  "ai_meta": 0,
  "ai_other": 0,
  "scroll_events": 0
}


In [8]:
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset
import itertools
import pandas as pd

login(token=userdata.get("HF_TOKEN"))

ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True)
filtered = ds['train'].filter(lambda row: str(row['report_date']).startswith('2026-03'))
rows = list(itertools.islice(filtered, 50000))
month_slice = pd.DataFrame(rows)

print(month_slice.shape)
print(month_slice.columns.tolist())
print(month_slice.head())


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(50000, 30)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available

## 2. Fields: feature / label / context / excluded

Feature: ctr, avg_position, impressions_90d, days_since_last_update — all knowable before any decision is made.
Label/proxy: an opportunity score derived from how far actual CTR falls below the expected CTR for that position tier.
Context: content_type, page_id — used to group and identify, not to predict on directly.
Excluded: any post-decision or outcome field like trend_pct or is_declining_label — these are derived from the label itself and would leak the answer, same lesson as notebook 02.

In [9]:
month_slice['ctr'] = month_slice['gsc_clicks'] / month_slice['gsc_impressions'].replace(0, pd.NA)
month_slice['avg_position'] = month_slice['gsc_avg_position']
features_df = month_slice[['client_hash_id','content_hash_id','report_date','ctr','avg_position','gsc_impressions']].copy()
print(features_df.head())

            client_hash_id           content_hash_id report_date    ctr  \
0  client_73cda7b4e4f265ea  content_b7e512995f79d5a6  2026-03-01    0.0   
1  client_73cda7b4e4f265ea  content_05597932fe4da067  2026-03-01    0.0   
2  client_73cda7b4e4f265ea  content_7a105f548d9c6916  2026-03-01  0.008   
3  client_73cda7b4e4f265ea  content_905aa32a0230694e  2026-03-01    0.0   
4  client_73cda7b4e4f265ea  content_a3ea9792f793ec72  2026-03-01    0.0   

   avg_position  gsc_impressions  
0      3.350000               20  
1      0.000000                1  
2      4.928000              125  
3      4.000000                7  
4      2.272727               11  


## 3. Verify it with queries (grain, counts, missing values, windows)

These three queries confirm the grain (no duplicate rows), the actual size and date range of my March 2026 slice, and where missing values exist — so my contract claims above are backed by real numbers, not assumptions.

In [10]:
dupes = month_slice.duplicated(subset=['client_hash_id','content_hash_id','report_date']).sum()
print("Duplicate rows:", dupes)
print("Row count:", month_slice.shape[0], "| Date span:", month_slice['report_date'].min(), "-", month_slice['report_date'].max())
print(month_slice[['ctr','avg_position','gsc_impressions']].isnull().sum())

Duplicate rows: 0
Row count: 50000 | Date span: 2026-03-01 - 2026-03-02
ctr                29229
avg_position       29229
gsc_impressions        0
dtype: int64


## 4. Data limits

This data can never tell me about seasonality beyond one month's window, so a holiday spike or a one-off traffic event won't show up. Some early rows may only have GSC-derived metrics with thinner history than recent months, and since I'm only using one month, window-overlap effects near month boundaries could be slightly off.

In [11]:
print(sorted(month_slice['report_date'].astype(str).str[:7].unique()))
print(month_slice['gsc_data_available'].value_counts())
print(month_slice['ga4_data_available'].value_counts())

['2026-03']
gsc_data_available
False    29229
True     20771
Name: count, dtype: int64
ga4_data_available
False    29675
True       841
Name: count, dtype: int64


## Self-check

I've stated the grain and time window in plain words, sorted every field I touch into feature/label/context/excluded with a reason, verified my claims with three real queries including a missing-values check, and named concrete limitations of my one-month slice.